#  UBC International Enrolment vs. Student Housing Capacity: An Analytical Study

This notebook presents an in-depth analysis of whether the number of **international students enrolling at UBC** is growing at a significantly faster pace than the growth in **on-campus residence bed capacity**.

###  Objective
To determine if UBC’s student housing infrastructure is keeping pace with rising international enrolment numbers.

###  What We Did
- **Scraped historical data** (2012–2023) on international enrolment and available residence beds
- Conducted **Exploratory Data Analysis (EDA)** to visualize trends over time
- Calculated **year-over-year growth rates** for both enrolment and residence capacity
- Performed a **one-tailed paired t-test** to statistically compare the two growth rates
- Provided a **conclusion based on hypothesis testing** to assess whether housing growth has lagged behind enrolment growth



The results from this analysis can guide recommendations around housing expansion and planning for international student support at UBC.


In [1]:
import pandas as pd # noqa: E303

### 📥 Loading the Dataset

The dataset used in this analysis was **constructed by scraping historical data** from official UBC sources, including:

- **UBC Enrolment Reports**  
- **Student Housing Facts & Figures** from the UBC Housing website  
- Archived charts, PDFs, and screenshots (available in the `raw/` data folder)

 Screenshots and source files are provided in the project directory under the `Data/` and `Raw/` folders for refernce.

In [2]:
enrol_data = pd.read_csv("Data/Enrolment_Vs_Housing_Data/Enrolment_Data.csv")
bed_data_read = pd.read_csv("Data/Enrolment_Vs_Housing_Data/housing-students-ubc-2007-2023.csv")
enrol_data.head()

,year,enrolment
0,2012,8438
1,2013,9371
2,2014,10903
3,2015,12117
4,2016,13182


In [3]:
enrol_data.shape

(12, 2)

In [4]:
enrol_data.describe()

,year,enrolment
count,12.000000,12.000000
mean,2017.500000,13897.250000
std,3.605551,3054.529279
min,2012.000000,8438.000000
25%,2014.750000,11813.500000
50%,2017.500000,15045.000000
75%,2020.250000,16274.500000
max,2023.000000,17243.000000


### 📈 Trend of International Student Enrolment at UBC (2012–2023)

The line chart below visualizes the trend in **international student enrolment** at UBC from 2012 to 2023. Each point represents the total number of international students enrolled in a given year.

### ✅ Conclusion from the Enrolment Trend

The visualization clearly shows a consistent upward trend in international student enrolment at UBC from 2012 to 2023, indicating rising global interest and demand for UBC programs. 

A noticeable dip in 2020 likely reflects the impact of the COVID-19 pandemic on global mobility. Despite this, enrolment has recovered and continued to grow in subsequent years.


In [5]:
#cleaning the data for the students beds available in residence starting from year 2012 to 2023
bed_data = bed_data_read[bed_data_read['Chart Year'] >= 2012]
bed_data = bed_data[["Chart Year","Student Beds (UBCV)"]]
bed_data.head()

,Chart Year,Student Beds (UBCV)
5,2012,9432
6,2013,10041
7,2014,9989
8,2015,10543
9,2016,11038


In [6]:
bed_data =bed_data.rename(columns={"Chart Year": 'year'})
combined_data = pd.merge(enrol_data, bed_data, on="year", how='inner')
combined_data

,year,enrolment,Student Beds (UBCV)
0,2012,8438,9432
1,2013,9371,10041
2,2014,10903,9989
3,2015,12117,10543
4,2016,13182,11038
5,2017,14685,11795
6,2018,15405,11795
7,2019,16098,12425
8,2020,15504,12425
9,2021,16804,12711


## 📊 UBC International Enrolment vs Residence Capacity (2012–2023)

The following line plot compares **UBC international student enrolment** and **student residence bed capacity** from 2012 to 2023. This visualization helps assess whether housing capacity is keeping pace with the growing number of international students.

### Key Observations from the UBC International Enrolment vs Residence Capacity (2012 - 2023) Graph:

  **Trends Over Time**:
   - International enrolment appears to show a general upward trend over the years, indicating UBC's growing popularity among international students.
   - Residence capacity likely increased as well, but potentially at a slower rate than enrolment growth.

 **Gap Analysis**:
   - The graph likely shows a widening gap between enrolment numbers and available residence beds over time.
   - This suggests housing availability may not have kept pace with the growth in international student numbers.

 **Implications**:
   - The growing gap may indicate increasing challenges for international students in finding on-campus housing.
   - This could lead to higher demand for off-campus housing and potentially increased housing costs in the surrounding area.


###  Calculating Year-over-Year Growth for Enrolment and Residence Beds

To analyze how international student enrolment and on-campus housing capacity have changed over time, we calculated the **year-over-year growth rates** for each metric using the following formula:

$$
\text{Growth Rate} = \frac{\text{Current Year Value} - \text{Previous Year Value}}{\text{Previous Year Value}}
$$

This formula was applied to both the `enrolment` and `beds` columns to create two new columns:
- `enrolment_growth`
- `bed_growth`

These columns represent the **change** from one year to the next, allowing us to compare the relative growth between student enrolment and available residence capacity.


In [7]:
combined_data.insert(2, "enrolment_growth" , (combined_data["enrolment"]/combined_data["enrolment"].shift(1))-1)
combined_data.insert(4, "bed_growth" , (combined_data["Student Beds (UBCV)"]/combined_data["Student Beds (UBCV)"].shift(1))-1)
combined_data

,year,enrolment,enrolment_growth,Student Beds (UBCV),bed_growth
0,2012,8438,NaN,9432,NaN
1,2013,9371,0.110571,10041,0.064567
2,2014,10903,0.163483,9989,-0.005179
3,2015,12117,0.111346,10543,0.055461
4,2016,13182,0.087893,11038,0.046951
5,2017,14685,0.114019,11795,0.068581
6,2018,15405,0.049030,11795,0.000000
7,2019,16098,0.044985,12425,0.053412
8,2020,15504,-0.036899,12425,0.000000
9,2021,16804,0.083849,12711,0.023018


###  Hypothesis Testing: Comparing Enrolment Growth vs. Residence Bed Growth

With year-over-year growth rates calculated, we now perform a **one-tailed paired t-test** to determine whether the growth in **international student enrolment** is significantly greater than the growth in **residence bed capacity**.

We define the population means as:

- **μ₁**: Mean year-over-year growth rate of international student enrolment  
- **μ₂**: Mean year-over-year growth rate of residence bed capacity

The hypotheses are:

- **Null Hypothesis (H₀)**:  
  $$\mu_1 \leq \mu_2$$  
  Enrolment growth is **less than or equal to** bed growth.

- **Alternative Hypothesis (H₁)**:  
  $$\mu_1 > \mu_2$$  
  Enrolment growth is **greater than** bed growth.

This test will help determine whether UBC’s residence expansion has kept pace with the increasing demand from international students.
